In [14]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model  # Not working properly so importing ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

from langchain_groq import ChatGroq
 

In [3]:
# Step 1: Load and split the documents 

loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)  # Semantic chunker as well we can use
chunks = splitter.split_documents(raw_docs)

In [5]:
# Step 2: Vector Store 
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks,embeddings)

## Step 3: Retriver 
retriever = vector_store.as_retriever(search_type="mmr",
                                      search_kwargs= {"k":5 , "fetch_k" :15 , "lambda_mult" : 0.7} 
                                      )
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001FC83F779E0>, search_type='mmr', search_kwargs={'k': 5, 'fetch_k': 15, 'lambda_mult': 0.7})

In [11]:
# Step 4: LLM and Prompt 

import os 
from  dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
llm = init_chat_model("openai:o4-mini")
llm

# Query Expansion

query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and usedful context.
Original Query : "{query}"

Expanded query:

""")

query_expansion_chain = query_expansion_prompt | llm | StrOutputParser()
query_expansion_chain



PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and usedful context.\nOriginal Query : "{query}"\n\nExpanded query:\n\n')
| ChatOpenAI(profile={'max_input_tokens': 200000, 'max_output_tokens': 100000, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001FC84EFEE40>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001FC84FA88F0>, root_client=<openai.OpenAI object at 0x000001FC84F9ABD0>, root_async_client=<open

In [12]:
query_expansion_chain.invoke({"query" : "Langchain Memory"})

'Expanded query:\n\n"LangChain memory" OR  \n"LangChain Memory Management" OR  \n"LangChain memory modules" OR  \n"ConversationBufferMemory" OR  \n"ConversationSummaryMemory" OR  \n"ChatMessageHistory" OR  \n"Long-term memory" OR  \n"Short-term memory" OR  \n"session memory" OR  \n"stateful LLM" OR  \n"persistent memory store" OR  \n"external memory backend" OR  \n"RedisMemory" OR  \n"FaissMemory" OR  \n"VectorStore memory" OR  \n"SQLMemory" OR  \n"MongoDBMemory" OR  \n"in-memory cache" OR  \n"prompt cache" OR  \n"RAG memory" OR  \n"retrieval-augmented generation" OR  \n"context window management" OR  \n"conversation state management" OR  \n"memory best practices" OR  \n"LangChain tutorials" OR  \n"LangChain Python SDK"'

In [13]:
answer_prompt = PromptTemplate.from_template(
"""
Answer the question based on the context below.

Context: {context}

Question: {input}
"""
)

# ?  what's alternate for create_stuff_documents_chain
document_chain = create_stuff_documents_chain(llm= llm, prompt=answer_prompt)

In [16]:
rag_pipeline = (
    RunnableMap({
        "input" : lambda x:x["input"] ,
        "context" : lambda x: retriever.invoke(query_expansion_chain.invoke({"query" : x["input"]}))
    })
    | document_chain
)

rag_pipeline

{
  input: RunnableLambda(...),
  context: RunnableLambda(...)
}
| RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
    context: RunnableLambda(format_docs)
  }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
  | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below.\n\nContext: {context}\n\nQuestion: {input}\n')
  | ChatOpenAI(profile={'max_input_tokens': 200000, 'max_output_tokens': 100000, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001FC84EFEE40>, async_client=<openai.resources.

In [20]:
query = {"input" : "What types of memory does langchain support?"}
print(query_expansion_chain.invoke({"query" : query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

Expanded query:
“What types of memory abstractions, modules, and storage backends does the LangChain framework support?  For example, what short-term or conversational chat memories (ConversationBufferMemory, ConversationBufferWindowMemory, ChatMessageHistory), long-term or summary memories (ConversationSummaryMemory, CombinedMemory), semantic or embedding-based/vector-store memories (VectorStoreRetrieverMemory with FAISS, Chroma, Pinecone, Weaviate, Milvus), key-value and JSON file memories (DictMemory, JSONFileMemory), relational/NoSQL datastore integrations (RedisMemory, SQLDatabaseMemory, PostgreSQL, MongoDB), knowledge-graph memories, and any other persistence layers or RAG (retrieval-augmented generation) modules supported by LangChain?”
✅ Answer:
 LangChain currently ships with two main “chat” memory modules:  
1. ConversationBufferMemory – keeps the full running buffer of past turns  
2. ConversationSummaryMemory – maintains a rolling summary of the conversation to stay within 

In [21]:
query = {"input" : "What types of memory does CrewAI support?"}
print(query_expansion_chain.invoke({"query" : query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

Expanded Query:

What types of memory does CrewAI support, including its various memory modules, storage backends and APIs? For example:

– Short-term/working memory (conversation history buffer, context window)  
– Long-term/persistent memory (knowledge base, factual store, event log)  
– Episodic memory (storing and recalling past interactions)  
– Semantic memory (concepts, embeddings, ontologies)  
– Retrieval-augmented memory (vector database, embedding store, RAG)  
– External memory plugins and integrations (SQL/NoSQL, Redis, Pinecone, Weaviate)  
– Memory management strategies (garbage collection, summarization, compression)  
– Memory API endpoints (read/write/update/delete)  
– Capacity and performance considerations  

Also consider synonyms and related terms: memory architecture, memory components, memory layers, memory backends, memory store, context retention, knowledge persistence, CrewAgent memory, Crew AI memory.
✅ Answer:
 The context never actually assigns any memory

In [ ]:
query = {"input" : "What types CrewAI Agents?"}
print(query_expansion_chain.invoke({"query" : query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

Expanded query:  
What types or categories of agents does the CrewAI platform provide? In other words, what classes, variants or models of CrewAI agents—such as autonomous scheduling agents, virtual crew assistants, workflow automation bots, AI-driven dispatchers, decision-support modules, analytics agents or integration connectors—are available? Include details on each agent’s capabilities, technical architecture, use cases, deployment options and configuration parameters.
✅ Answer:
 CrewAI agents are organized by role within the crew workflow. The three primary agent types are:

1. Researcher – gathers information, runs analyses, and compiles insights.  
2. Planner – outlines strategies, breaks the overall goal into tasks, and sequences them.  
3. Executor – carries out individual tasks (e.g. writing, coding, outreach) using the tools provided.  

Each agent type has its own purpose, goal, and toolset but collaborates with the others to achieve the crew’s overarching objective.


: 